# Экзамен по курсу «Анализ данных в Python»

## Плей-офф НХЛ сезона 2025/2026

Используемые библиотеки:
`pandas` - работа с таблицами
`numpy` - математика
`matplotlib` и `seaborn` - графики
`rink`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rink import plot_rink
pd.set_option('display.precision', 3)

events = pd.read_excel('NHL-playoffs-2026.xlsx', index_col=0)
events.shape

Посмотрим на первые 10 строк и несколько столбцов - дата, команды, оставшееся время периода и тип события:

In [ ]:
events.iloc[:10, [2, 3, 4, 9, 12]]

Посмотрим, какие типы событий встречаются и сколько их:

In [ ]:
events['Play Type'].value_counts()

Чаще всего в записях встречаются `hit`, вбрасывания `faceoff` и броски по воротам `shot-on-goal`. Голов `goal` 486. Событий `game-end` 82 — по одному на каждый матч.
При изучении голов обнаружилась ошибка в данных: у некоторых голов количество игроков на льду равно нулю, поля с фамилиями игроков и вратарей пустые. Посмотрим, что это за голы:

In [ ]:
broken = (events['Play Type'] == 'goal') & (events["# of Home Team's Players on the Ice"] == 0)
print('Голов с пустым списком игроков на льду:', broken.sum())
events[broken]['Period'].value_counts()

Все такие голы забиты в овертаймах (период 4 и 5) - то есть это победные голы, после которых игра заканчивается. Ворота при этом не могут быть пустыми, тк в овертайме плей-офф вратарей не снимают.
Это важно для заданий, где нужно знать вратаря, пропустившего гол. Поэтому для таких голов возьмём вратарей из предыдущего события той же игры - скорее всего за несколько секунд до гола в воротах стояли те же вратари.

In [ ]:
prev = events.shift(1)
events.loc[broken, "Home Team's Goalie"] = prev.loc[broken, "Home Team's Goalie"]
events.loc[broken, "Away Team's Goalie"] = prev.loc[broken, "Away Team's Goalie"]

events[broken][['Game ID', 'Period', "Home Team's Goalie", "Away Team's Goalie"]].head()

# 2. Сводная таблица по играм

Чтобы собрать итоги каждого матча, берем событие `game-end`- в нём уже записан итоговый счёт и номер последнего периода.

По номеру последнего периода определяем исход:
- период 3 — победитель определился в основное время: `Final`;
- период 4 — понадобился один овертайм: `Final/OT`;
- период 5 — два овертайма: `Final/2OT`, и так далее

`Game ID` делаем индексом таблицы.

In [ ]:
ends = events[events['Play Type'] == 'game-end']

games = ends[['Game ID', 'Date', 'Away Team', 'Home Team',
              'Cumulated Away Score', 'Cumulated Home Score', 'Period']]
games = games.rename(columns={'Cumulated Away Score': 'Away Score',
                              'Cumulated Home Score': 'Home Score'})
games = games.set_index('Game ID')


def outcome(period):
    if period == 3:
        return 'Final'
    elif period == 4:
        return 'Final/OT'
    else:
        return 'Final/' + str(period - 3) + 'OT'


games['Outcome'] = games['Period'].apply(outcome)
games = games.drop(columns='Period')

games.columns

In [ ]:
games.iloc[-5:, -4:]

Проверим, что итоговые счета не потерялись: сумма всех забитых шайб по таблице `games` должна совпадать с числом событий `goal`.

In [ ]:
print('Голов по таблице:', games['Away Score'].sum() + games['Home Score'].sum())
print('Событий goal:', (events['Play Type'] == 'goal').sum())

Числа совпадают - таблица собрана правильно. Посмотрим, сколько матчей закончилось в основное время, а сколько в овертаймах:

In [ ]:
games['Outcome'].value_counts()

Из 82 матчей 60 закончились в основное время, а 22 потребовали овертайма. Из них 4 матча шли до второго овертайма. В плей-офф ничьих не бывает, и овертаймы играются до первой заброшенной шайбы.

# 3. Функция `gameInfo()`

Функция принимает номер игры, находит строку в таблице `games` и собирает из неё строку вида «Хозяева - Гости (голы хозяев - голы гостей, исход, дата)».

In [ ]:
def gameInfo(gameId):
    g = games.loc[gameId]
    return (g['Home Team'] + ' - ' + g['Away Team'] + ' ('
            + str(g['Home Score']) + ' - ' + str(g['Away Score']) + ', '
            + g['Outcome'] + ', ' + g['Date'] + ')')

## 4. Общее число голов в матче

Складываем голы гостей и хозяев и смотрим описательные статистики с помощью `.describe()`: число матчей, среднее, стандартное отклонение, минимум, квартили и максимум.

In [ ]:
games['Total Score'] = games['Away Score'] + games['Home Score']
games['Total Score'].describe()

В среднем в матче забивали почти 6 шайб (5.93), медиана — тоже 6. У половины матчей общий счёт был от 5 до 7 голов (между первым и третьим квартилями). Самый нерезультативный матч — всего 1 гол, самый результативный — 15 голов. Среднее и медиана почти совпадают, но максимум (15) намного дальше от середины, чем минимум (1), — это значит, что справа у распределения есть редкие очень результативные матчи.

## 5. Гистограмма числа голов в матче

Число голов - целое, используем `sns.countplot()`: он рисует отдельный столбец для каждого значения. Параметр `order` задаёт значения от 1 до макс, чтобы на оси были видны все значения.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=games, x='Total Score',
              order=range(1, games['Total Score'].max() + 1), color='tab:blue')
plt.xlabel('Число голов')
plt.ylabel('Число матчей')
plt.yticks(range(0, 21, 3))  
plt.title('Гистограмма числа голов в одном матче')
plt.show()

**Комментарий по заданию:**
Чаще всего в матче забивали 5 голов (19 матчей), затем 6 и 7 голов (по 16 матчей). Вместе матчи с 5–7 голами составляют больше половины всех игр. Матчей с 1–2 голами очень мало (3), матчей с 10 и более голами тоже мало (4). Распределение похоже на колокол с центром около 6, но с длинным правым хвостом: есть один матч с 15 голами, значения 12–14 не встретились ни разу. 3 гола встречаются почти в два раза чаще, чем 4 (9 матчей против 5). Это может быть случайностью, тк исследование проводим по малой выборке.

# 6. Самые нерезультативные и самые результативные игры
Найдем минимальное и максимальное значение `Total Score`, выбираем все игры с таким значением. Для каждой игры выводим информацию с помощью `gameInfo()`.

In [ ]:
min_score = games['Total Score'].min()
low_games = games[games['Total Score'] == min_score]

low_list = []
for game_id in low_games.index:
    low_list.append(gameInfo(game_id))
low_list

In [ ]:
max_score = games['Total Score'].max()
high_games = games[games['Total Score'] == max_score]

high_list = []
for game_id in high_games.index:
    high_list.append(gameInfo(game_id))
high_list

Самых нерезультативных матчей два, оба закончились со счётом 1:0 в овертайме: Tampa Bay обыграла Montréal, Philadelphia — Pittsburgh. За основное время ни одна команда не забила. Самый результативный матч — победа Colorado Avalanche над Minnesota Wild 9:6, забили 15 шайб.

## 7. Число игроков, участвовавших в каждой игре

Фамилии игроков на льду записаны в 12 столбцах (по 6 у гостей и хозяев) и ещё в двух столбцах с вратарями. Составим список названий этих столбцов.
Дальше используем метод `.melt()`: он складывает 14 столбцов в один длинный столбец `Player`, где каждой строке соответствует пара «игра — игрок». Пустые значения удаляем через `.dropna()`, а повторы одной и той же пары «игра — игрок» — через `.drop_duplicates()`. В результате получается таблица `appearances`: кто из игроков в каких играх выходил на лёд.

In [ ]:
player_cols = []
for i in range(1, 7):
    player_cols.append("Away Team's Player on the Ice #" + str(i))
    player_cols.append("Home Team's Player on the Ice #" + str(i))
player_cols.append("Away Team's Goalie")
player_cols.append("Home Team's Goalie")

appearances = events[['Game ID'] + player_cols].melt(id_vars='Game ID', value_name='Player')
appearances = appearances[['Game ID', 'Player']].dropna().drop_duplicates()
appearances.head()

Теперь число игроков в каждой игре — это просто количество строк для этой игры в таблице `appearances`.

In [ ]:
games['Player Count'] = appearances.groupby('Game ID').size()

game_info_list = []
for game_id in games.index:
    game_info_list.append(gameInfo(game_id))
games['Game Info'] = game_info_list

max_players = games['Player Count'].max()
games[games['Player Count'] == max_players][['Game Info', 'Player Count']]

Посмотрим, почему в этой игре игроков больше обычного: выведем вратарей каждой команды по периодам.

In [ ]:
max_id = games['Player Count'].idxmax()
one_game = events[events['Game ID'] == max_id]
print('Вратари гостей:', one_game["Away Team's Goalie"].dropna().unique())
print('Вратари хозяев:', one_game["Home Team's Goalie"].dropna().unique())

In [ ]:
games['Player Count'].describe()

In [ ]:
games['Player Count'].value_counts()

**Комментарий.** В 73 матчах из 82 на лёд выходили ровно 38 игроков. Это объясняется правилами НХЛ: команда заявляет на матч 18 полевых игроков и 2 вратарей, но запасной вратарь обычно не играет. Получается 2 × (18 + 1) = 38. Если одна из команд меняла вратаря по ходу игры, игроков становится 39 (так было в 8 матчах).

Больше всего игроков (40) было в матче Buffalo Sabres в гостях у Montréal Canadiens, который закончился разгромом 8:3. В нём вратарей меняли обе команды: у Buffalo играли Alex Lyon и Ukko-Pekka Luukkonen, у Montréal — Jakub Dobes и Jacob Fowler.

## 8. Расстояние от точки броска до ворот

Броски трёх типов: `shot-on-goal` и `goal` (точные броски) и `missed-shot` (мимо ворот).

**В какие ворота бросает команда.** Поле `Home Team's Defending Side` говорит, с какой стороны ворота хозяев. Если хозяева защищают левые ворота (`left`), то сами они бросают в правые (X = 89), а гости — в левые (X = −89). Если `right` — наоборот. Кто бросает, понятно из сравнения `Team Making the Play` с `Home Team`. Команды меняются воротами каждый период, и это поле меняется вместе с ними, поэтому его нужно смотреть для каждого события отдельно.

**Расстояние до створа.** Ворота имеют ширину 6 футов: от y = −3 до y = 3. Если бросок сделан напротив ворот (−3 ≤ y ≤ 3), то расстояние — это просто |x − X|. Если сбоку, то считаем расстояние до ближайшей штанги (точки (X, −3) или (X, 3)) по теореме Пифагора.

Напишем две функции, которые работают с одной строкой таблицы, и применим их ко всем броскам через `.apply(..., axis=1)`.

In [ ]:
def goal_x(row):
    home_shoots = row['Team Making the Play'] == row['Home Team']
    if row["Home Team's Defending Side"] == 'left':
        if home_shoots:
            return 89
        else:
            return -89
    else:
        if home_shoots:
            return -89
        else:
            return 89


def shot_length(row):
    x = row['X Coordinate on the Ice Rink']
    y = row['Y Coordinate on the Ice Rink']
    X = goal_x(row)
    if y < -3:
        return np.sqrt((x - X) ** 2 + (y + 3) ** 2)
    elif y <= 3:
        return abs(x - X)
    else:
        return np.sqrt((x - X) ** 2 + (y - 3) ** 2)


is_shot = events['Play Type'].isin(['shot-on-goal', 'goal', 'missed-shot'])
events.loc[is_shot, 'Shot Length'] = events[is_shot].apply(shot_length, axis=1)

events[is_shot][['Game ID', 'Play Type', 'Shot Length']]

Всего 7266 бросков. Описательные статистики длины броска:

In [ ]:
events.loc[is_shot, 'Shot Length'].describe()

Средняя длина броска - около 35 футов (примерно 11 м), медиана — 32 фута. Три четверти бросков сделаны с расстояния меньше 48 футов - из зоны нападения. Самый дальний бросок - 188 футов. Минимальное расстояние 0 - броски с самой линии ворот.

## 9. Распределение длины точных и неточных бросков

Добавим столбец `Shot` со значениями `on goal` (точный бросок) и `missed` (мимо). Неточных и точных бросков разное количество, поэтому если рисовать обычные гистограммы с количеством, то высоты столбиков сравнивать нельзя. Чтобы сравнивать именно распределения, используем в `sns.displot()`:
- `stat='density'` - по вертикали плотность, а не количество;
- `common_norm=False` - каждая группа нормируется отдельно (площадь под каждой гистограммой равна 1).

`kde=True` добавляет сглаженную кривую плотности.

In [ ]:
shots = events[is_shot].copy()
shots['Shot'] = 'on goal'
shots.loc[shots['Play Type'] == 'missed-shot', 'Shot'] = 'missed'

print(shots['Shot'].value_counts())

g = sns.displot(data=shots, x='Shot Length', hue='Shot', hue_order=['missed', 'on goal'],
                stat='density', common_norm=False, kde=True, height=5, aspect=1.5)
g.set_axis_labels('Shot length, feet', 'Frequency')
plt.show()

**Выводы по графику:**
1. Формы двух распределений похожи: большинство бросков - и точных, и неточных, делается с расстояния до 70 футов, то есть из зоны нападения
2. С очень близкого расстояния (до 10 футов) и со средних дистанций (30–40 футов) чаще попадают в створ: оранжевые столбцы там выше синих. Вблизи ворот промахнуться трудно, а броски с 30–40 футов часто делаются в расчёте на добивание.
3. С 45–70 футов относительно больше неточных бросков: чем дальше от ворот, тем труднее попасть в створ.
4. Дальше 75 футов почти все броски точные: у оранжевой гистограммы длинный хвост до 188 футов, а синих бросков там почти нет.

**Почему такой длинный хвост точных бросков со своей половины?** Расстояние больше 89 футов означает, что бросок сделан из-за центральной линии, со своей половины площадки. Такой бросок почти никогда не делают, чтобы забить. Есть две причины, почему такие броски часто попадают в данные именно как точные:
- **Броски в пустые ворота.** В конце третьего периода проигрывающая команда меняет вратаря на шестого полевого игрока, и тогда соперник может бросать прямо из своей зоны. Если шайба летит в створ, это бросок по воротам или гол, а промахи из своей зоны часто фиксируются как проброс, а не как бросок мимо.
- **Вбросы шайбы в зону («dump-in»).** Если такая шайба долетела до ворот и вратарь её остановил, это записывается как бросок в створ (`shot-on-goal`). Похожие вбросы мимо ворот бросками обычно не считаются.

Проверим первое объяснение по данным: когда были забиты голы со своей половины.

In [ ]:
long_shots = shots[shots['Shot Length'] > 89]
print('Бросков со своей половины:', len(long_shots))
print(long_shots['Play Type'].value_counts())

long_goals = long_shots[long_shots['Play Type'] == 'goal']
long_goals[['Game ID', 'Period', 'Elapsed Time', 'Team Making the Play', 'Shot Length']]

Со своей половины площадки было 251 бросок, из них 202 точных, 19 голов и только 30 промахов. 17 голов из 19 забиты в третьем периоде в последние 4 минуты (время с начала периода от 16:24 до 19:56). Именно в это время проигрывающая команда снимает вратаря, так что это наверняка голы в пустые ворота. Только два гола забиты раньше: во втором периоде с центральной линии и в третьем периоде на 15-й минуте. Это редкие дальние голы при вратаре в воротах.

Есть ещё одна особенность - у многих из этих голов в поле вратаря пропустившей команды всё равно записан вратарь. Видимо, состав на льду для гола записан на момент следующего вбрасывания, когда вратарь уже вернулся в ворота. Поэтому пустые ворота мы определяем по времени гола, а не по полю с вратарём.

## 10. Доля точных бросков у каждой команды в каждой игре

Незаблокированные броски - те же три типа, что в пункте 8: `shot-on-goal`, `goal` и `missed-shot`. Точные — `shot-on-goal` и `goal`. Для каждой пары (игра, команда) считаем количество точных и количество всех незаблокированных бросков с помощью `.groupby().size()`, а потом делим одно на другое.

In [ ]:
on_target = shots[shots['Shot'] == 'on goal']

numshots = pd.DataFrame()
numshots['Shots on Goal'] = on_target.groupby(['Game ID', 'Team Making the Play']).size()
numshots['All Shots'] = shots.groupby(['Game ID', 'Team Making the Play']).size()
numshots['Share of Shots on Goal'] = numshots['Shots on Goal'] / numshots['All Shots']

numshots['Share of Shots on Goal']

In [ ]:
numshots['Share of Shots on Goal'].describe()

Чтобы понять, связана ли точность с результатом, отметим для каждой команды в каждой игре, выиграла она или нет, и сравним среднюю долю точных бросков у победителей и у проигравших.

In [ ]:
winners = []
for game_id in numshots.index.get_level_values('Game ID'):
    g = games.loc[game_id]
    if g['Home Score'] > g['Away Score']:
        winners.append(g['Home Team'])
    else:
        winners.append(g['Away Team'])

teams = numshots.index.get_level_values('Team Making the Play')
numshots['Win'] = (teams == winners)

numshots.groupby('Win')['Share of Shots on Goal'].describe()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=numshots, x='Share of Shots on Goal', hue='Win', stat='density',
             common_norm=False, kde=True, bins=15)
plt.xlabel('Доля точных бросков в матче')
plt.ylabel('Плотность')
plt.show()

**Выводы по графику:**
- В среднем команда попадает в створ примерно в 64 % незаблокированных бросков (и среднее, и медиана 0.639). То есть примерно 2 броска из 3, которые не заблокировали полевые игроки, летят в ворота, а третий проходит мимо.
- Разброс заметный: в половине матчей доля от 0.58 до 0.69, а крайние значения — от 0.35 до 0.86. В отдельном матче доля сильно зависит от случая, потому что за игру команда делает всего 40–50 незаблокированных бросков.
- У победителей средняя доля точных бросков не выше, чем у проигравших (0.632 против 0.646), а распределения на графике почти совпадают. Значит, точность сама по себе вряд ли определяет победу. Важнее, откуда бросают, сколько всего бросков делает команда и как играет вратарь соперника. Возможно есть и обратный эффект: проигрывающая команда в конце матча чаще бросает с любой позиции, чтобы как-нибудь отыграться, а лидирующая играет осторожнее.

## 11. Команды с самой высокой и самой низкой средней точностью

Усредняем долю точных бросков по всем матчам каждой команды и выбираем по пять лучших (`.nlargest(5)`) и худших (`.nsmallest(5)`). Рядом полезно видеть, сколько матчей сыграла команда, потому что у команд, рано вылетевших из плей-офф, среднее посчитано всего по 4–6 играм.

In [ ]:
avgnumshots = numshots.groupby('Team Making the Play')['Share of Shots on Goal'].mean()
avgnumshots.nlargest(5)

In [ ]:
avgnumshots.nsmallest(5)

In [ ]:
games_played = games['Home Team'].value_counts() + games['Away Team'].value_counts()
games_played.index.name = 'Team Making the Play'

team_table = pd.DataFrame()
team_table['Share of Shots on Goal'] = avgnumshots
team_table['Games'] = games_played
team_table.sort_values('Share of Shots on Goal', ascending=False)

**Комментарий по таблице:**
- Самые точные в среднем — Edmonton Oilers (70.6 % бросков в створ), Philadelphia Flyers и Dallas Stars. Самые неточные — Pittsburgh Penguins (55.4 %) и Utah Mammoth.
- Разница между командами небольшая: почти все команды попадают в створ в 60–70 % незаблокированных бросков.
- Средняя точность почти не связана с успехом в плей-офф. Например, Edmonton, Dallas и Philadelphia с самой высокой точностью выбыли довольно рано (6 и 10 матчей). А Carolina Hurricanes, с одной из самых низких долей (61.5 %), выиграли в финале, сыграв суммарно 19 матчей.
- У команд, сыгравших 4–6 матчей, среднее посчитано по маленькой выборке, поэтому их места в рейтинге могут быть неоправданно завышены или занижены.

## 12. Среднее число бросков в створ и голов за игру у каждой команды

Броски в створ ворот - это `shot-on-goal` и `goal`. Число бросков и голов каждой команды делим на количество сыгранных ею матчей `games_played`

In [ ]:
goals = events[events['Play Type'] == 'goal']

avgshots = pd.DataFrame()
avgshots['Shot'] = on_target.groupby('Team Making the Play').size() / games_played
avgshots['Goal'] = goals.groupby('Team Making the Play').size() / games_played
avgshots

## 13. Связь бросков в створ и голов

Рисуем точки (одна точка одна команда) и линию регрессии с доверительной областью функцией `sns.regplot()`. Подписываем все команды, чтобы было видно, кто где находится. Дополнительно посчитаем коэффициент корреляции.

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(data=avgshots, x='Shot', y='Goal')
for team in avgshots.index:
    plt.text(avgshots.loc[team, 'Shot'], avgshots.loc[team, 'Goal'] + 0.04, team,
             fontsize=8, ha='center')
plt.xlabel('Average number of shots on goal per game')
plt.ylabel('Average number of goals per game')
plt.show()

In [ ]:
avgshots.corr()

**Комментарий:** 
Связь есть, но не сильная. Линия регрессии идёт вверх: команды, которые в среднем больше бросают в створ, в среднем больше и забивают. Коэффициент корреляции 0.52 — это умеренная положительная связь.

При этом доверительная область вокруг линии очень широкая, потому что точек всего 16, и несколько команд сильно отклоняются от линии:
- **Ottawa Senators** и **Los Angeles Kings** забивали всего 1.25 гола за игру, хотя Ottawa бросала в створ не так уж мало (27.5 бросков). Это команды, быстро вылетевшие в первом раунде (по 4 матча), — им «не везло» с реализацией;
- **Montréal Canadiens** меньше всех бросали (23.3 броска за игру), но забивали 2.8 гола — высокая реализация моментов;
- **Vegas Golden Knights** и **Minnesota Wild** забивали намного больше, чем предсказывает линия по числу бросков;
- **Dallas Stars**, наоборот, бросали много (31 бросок), но забивали только 2.5.

Вывод: число бросков в створ помогает объяснить число голов, но сильно влияют и другие факторы типа качества бросков (с какого расстояния и каких позиций), уровень вратарей соперника и случайность на маленьком числе матчей.

## 14. Матч с наибольшим числом ударов по воротам

Теперь считаем все удары: точные, мимо и заблокированные (`blocked-shot`). Для каждой игры считаем количество таких событий и находим игру с максимумом через `.idxmax()` (возвращает номер игры).

In [ ]:
all_shot_types = ['goal', 'shot-on-goal', 'blocked-shot', 'missed-shot']
all_shots = events[events['Play Type'].isin(all_shot_types)]

shots_per_game = all_shots.groupby('Game ID').size()
idxmax = shots_per_game.idxmax()
print('Ударов в этой игре:', shots_per_game.max())
gameInfo(idxmax)

Больше всего ударов (208) было в матче первого раунда Carolina Hurricanes — Ottawa Senators 20 апреля, который закончился победой Carolina 3:2 во втором овертайме - игра шла почти пять периодов вместо трёх, поэтому бросков набралось больше обычного. 
Посмотрим, как удары распределились по командам и исходам:

In [ ]:
all_shots[all_shots['Game ID'] == idxmax].groupby(['Team Making the Play', 'Play Type']).size().unstack()

Хозяева (Carolina) нанесли немного больше ударов, чем Ottawa: 109 против 99. В створ (броски по воротам плюс голы) 49 против 39. Количество заблокированных и неточных ударов у команд почти одинаковое.

## 15. Карта ударов в этом матче

Нужно нарисовать удары гостей так, будто они всегда бьют в левые ворота, а удары хозяев в правые. Команды меняются воротами каждый период, поэтому часть координат надо отразить: (x, y) в (−x, −y).

Посмотрим, как в этом матче менялась сторона ворот хозяев по периодам:

In [ ]:
match = all_shots[all_shots['Game ID'] == idxmax].copy()
match.groupby('Period')["Home Team's Defending Side"].unique()

В нечётных периодах хозяева защищают правые ворота, то есть бьют в левые, а гости в правые. Это противоположно тому, что нам нужно, поэтому, как и сказано в задании, координаты всех ударов в нечётных периодах отражаем, а в чётных оставляем как есть.

Цвет точки показывает исход удара (`Play Type`), а форма маркера - команду: треугольник вправо для хозяев (бьют в правые ворота) и треугольник влево для гостей.

In [ ]:
match['x'] = match['X Coordinate on the Ice Rink']
match['y'] = match['Y Coordinate on the Ice Rink']

odd = match['Period'] % 2 == 1
match.loc[odd, 'x'] = -match.loc[odd, 'x']
match.loc[odd, 'y'] = -match.loc[odd, 'y']

home_team = games.loc[idxmax, 'Home Team']
away_team = games.loc[idxmax, 'Away Team']

plt.figure(figsize=(12, 5.5))
plot_rink()
ax = sns.scatterplot(data=match, x='x', y='y',
                     hue='Play Type', hue_order=all_shot_types,
                     palette={'goal': 'red', 'shot-on-goal': 'm',
                              'blocked-shot': 'blue', 'missed-shot': 'green'},
                     style='Team Making the Play',
                     markers={home_team: '>', away_team: '<'},
                     s=40)
sns.move_legend(ax, 'upper center', fontsize=8, title=None)
plt.axis('equal')
plt.axis('off')
plt.title('Удары по воротам в матче ' + gameInfo(idxmax))
plt.tight_layout()
plt.show()

**Комментарий к рисунку:** Почти все удары нанесены из зоны нападения, между синей линией и воротами. Больше всего точек в области перед воротами между кругами вбрасывания и у самых ворот тк оттуда легче забить. Все голы забиты с близкого расстояния, из слота или почти с линии ворот. Заблокированные удары чаще бывают со средних дистанций, у синей линии и у кругов вбрасывания тк защитникам проще перекрыть дальний бросок. Удары со своей половины площадки единичные. Справа (удары Carolina) точек немного больше, чем слева (удары Ottawa), особенно пурпурных бросков в створ. Это совпадает с таблицей выше.

## 16. Лучшие снайперы плей-офф

Автор гола записан в поле `Player in the Event #1` у событий `goal`. Считаем, сколько раз встречается каждый игрок. Чтобы было понятно, за кого играет игрок, считаем пары «игрок - команда».

Если несколько игроков набрали одинаковое число голов, обычный `.head(10)` мог бы отрезать часть из них случайным образом. Поэтому используем `.nlargest(10, keep='all')` он оставляет всех игроков, у которых результат равен десятому месту.

In [ ]:
goal_scorers = goals[['Player in the Event #1', 'Team Making the Play']]
goal_scorers = goal_scorers.rename(columns={'Player in the Event #1': 'Player',
                                            'Team Making the Play': 'Team'})

goal_count = goal_scorers.value_counts()
goal_count.name = 'Goals'
goal_count.nlargest(10, keep='all')

Лучший снайпер плей-офф  Бретт Хауден из Vegas Golden Knights с 14 голами, за ним идут Павел Дорофеев из Вегаса, 12 голов  и Логан Станковен из Каролины с 11 голами. На 10-м месте оказались сразу пять игроков с 7 голами, поэтому в списке 12 человек.

Почти весь список это игроки двух финалистов - Vegas и Carolina. Отчасти это потому, что финалисты сыграли больше всего матчей (22 и 19), а значит, у их игроков было больше возможностей забить.

## 17. Лучшие ассистенты

Голевые передачи записаны в полях `Player in the Event #2` (первый ассистент) и `Player in the Event #3` (второй ассистент). Берём оба столбца, переименовываем их одинаково и ставим друг под другом через `pd.concat()`. Пустые значения (гол без передач или с одной передачей) удаляем.

In [ ]:
assist1 = goals[['Player in the Event #2', 'Team Making the Play']]
assist1 = assist1.rename(columns={'Player in the Event #2': 'Player',
                                  'Team Making the Play': 'Team'})
assist2 = goals[['Player in the Event #3', 'Team Making the Play']]
assist2 = assist2.rename(columns={'Player in the Event #3': 'Player',
                                  'Team Making the Play': 'Team'})

assists = pd.concat([assist1, assist2]).dropna()

assist_count = assists.value_counts()
assist_count.name = 'Assists'
assist_count.nlargest(10, keep='all')

Больше всего голевых передач у двух игроков Vegas: Jack Eichel (20) и Mitch Marner (19). Дальше с большим отрывом идут Lane Hutson (Montréal) и Jackson Blake (Carolina) - по 13. Снова видно, что лидеры статистики в основном из команд, сыгравших много матчей: Vegas, Carolina и Montréal (полуфиналист, 19 матчей).

## 18. Лучшие бомбардиры (гол + пас)

Объединяем голы и передачи в одну таблицу. У кого-то нет голов или нет передач там получатся пропуски, их заменяем нулями. Очки это сумма голов и передач. При равенстве очков выше ставим того, у кого больше голов.

In [ ]:
points = pd.DataFrame({'Goals': goal_count, 'Assists': assist_count})
points = points.fillna(0).astype(int)
points['Points'] = points['Goals'] + points['Assists']

points = points.sort_values(['Points', 'Goals'], ascending=False)
points.head(10)

Лучший бомбардир плей-офф - Mitch Marner (Vegas): 29 очков (10 голов + 19 передач). У него большой отрыв от второго места, Jack Eichel (22 очка, почти все - передачи). Дальше идут игроки Carolina Jackson Blake (20) и Taylor Hall (19). В десятке 5 игроков Vegas и 4 игрока Carolina. Интересно, что лучший снайпер Brett Howden по очкам только пятый: у него много голов, но мало передач.

## 19. Больше всего бросков в створ за игру

Бросок в створ - события `shot-on-goal` и `goal`, бросающий записан в `Player in the Event #1`. Число бросков каждого игрока делим на число игр, в которых он выходил на лёд (считаем по таблице `appearances` из пункта 7).

In [ ]:
games_by_player = appearances['Player'].value_counts()

player_shots = pd.DataFrame()
player_shots['Shots'] = on_target['Player in the Event #1'].value_counts()
player_shots['Games'] = games_by_player
player_shots['Shots per Game'] = player_shots['Shots'] / player_shots['Games']

player_shots.sort_values('Shots per Game', ascending=False).head(10)

Больше всего бросков в створ за игру делал Jason Robertson (Dallas Stars) - ровно 5 за матч. За ним Dylan Guenther (Utah, 4.5) и Nikita Kucherov (Tampa Bay, 4.4). Все игроки десятки сыграли не меньше 4 матчей, так что случайных «одноматчевых» лидеров здесь нет.

В этом рейтинге, в отличие от рейтинга голов, много игроков команд, выбывших рано (Dallas, Utah, Tampa Bay, Los Angeles, Boston). Среднее за игру не зависит от числа сыгранных матчей, поэтому оно лучше показывает активность игрока, чем общие суммы. Из тех, кто играл долго, выделяются Matt Boldy (Minnesota, 48 бросков за 11 матчей) и Nathan MacKinnon (Colorado, 47 бросков за 13 матчей).

## 20. Больше всего силовых приёмов за игру

Делаем то же самое для событий `hit`: игрок, проводящий силовой приём, записан в `Player in the Event #1`.

In [ ]:
hits = events[events['Play Type'] == 'hit']

player_hits = pd.DataFrame()
player_hits['Hits'] = hits['Player in the Event #1'].value_counts()
player_hits['Games'] = games_by_player
player_hits['Hits per Game'] = player_hits['Hits'] / player_hits['Games']

player_hits.sort_values('Hits per Game', ascending=False).head(10)

Здесь в десятку попали игроки, сыгравшие всего 1 матч (Alex Turcotte и Alexander Petrovic - по 6 силовых приёмов в единственной игре) и 3 матча. Для них среднее посчитано по слишком маленькому числу игр и может быть случайным числом. Каждая серия плей-офф длится минимум 4 игры, поэтому дополнительно посмотрим только на игроков, сыгравших не меньше 4 матчей:

In [ ]:
regular_hitters = player_hits[player_hits['Games'] >= 4]
regular_hitters.sort_values('Hits per Game', ascending=False).head(10)

Среди игроков, сыгравших хотя бы 4 матча, больше всего силовых приёмов за игру проводил Lawson Crouse - 6.3 за матч. Дальше идут Яков Тренин (5.7) и Dylan Cozens (5.5). Отдельно стоит отметить Ивана Барбашёва из Vegas: 110 силовых приёмов за 22 матча (5 за игру). Он держит высокий темп на протяжении всего плей-офф, вплоть до финала. В рейтинге в основном нападающие, для которых силовая борьба - важная часть игровой роли. Звёзды-бомбардиры из пунктов 16–18 сюда не попали.

## 21. Вратари, хотя бы раз не пропустившие ни одного гола

Логика такая:
1. Составляем список всех пар «игра - вратарь» для каждого вратаря, который хотя бы раз был на льду в этой игре (поля `Away Team's Goalie` и `Home Team's Goalie`). Сюда попадают и вратари, которые выходили на замену.
2. Для каждого гола определяем, кто стоял в воротах пропустившей команды: если забили хозяева, то пропустил вратарь гостей, и наоборот. Если поле с вратарём пустое, значит ворота были пустыми (вратаря заменили на полевого игрока) - такой гол вратарю не засчитывается.
3. Оставляем те пары «игра - вратарь», для которых в игре не было ни одного пропущенного этим вратарём гола.

Чтобы удобно сравнивать пары, склеиваем номер игры и фамилию в одну строку-ключ.

In [ ]:
home_goalies = events[['Game ID', "Home Team's Goalie"]].rename(columns={"Home Team's Goalie": 'Goalie'})
away_goalies = events[['Game ID', "Away Team's Goalie"]].rename(columns={"Away Team's Goalie": 'Goalie'})
goalie_games = pd.concat([home_goalies, away_goalies]).dropna().drop_duplicates()

conceded = goals[['Game ID', 'Team Making the Play', 'Home Team',
                  "Home Team's Goalie", "Away Team's Goalie"]].copy()
conceded['Goalie'] = conceded["Home Team's Goalie"]
home_scored = conceded['Team Making the Play'] == conceded['Home Team']
conceded.loc[home_scored, 'Goalie'] = conceded.loc[home_scored, "Away Team's Goalie"]
conceded = conceded[['Game ID', 'Goalie']].dropna().drop_duplicates()

goalie_games['key'] = goalie_games['Game ID'].astype(str) + ' ' + goalie_games['Goalie']
conceded['key'] = conceded['Game ID'].astype(str) + ' ' + conceded['Goalie']

clean_sheets = goalie_games[~goalie_games['key'].isin(conceded['key'])].copy()

info_list = []
for game_id in clean_sheets['Game ID']:
    info_list.append(gameInfo(game_id))
clean_sheets['Game Info'] = info_list

clean_sheets[['Goalie', 'Game Info']].sort_values('Goalie')

In [ ]:
sorted(clean_sheets['Goalie'].unique())

**Комментарий.** Хотя бы в одном матче не пропустили ни одного гола 6 вратарей:
- Frederik Andersen (Carolina) - 3 раза, больше всех. Это одна из причин успеха Carolina в плей-офф;
- Dan Vladar (Philadelphia) - 2 раза, в обоих матчах против Pittsburgh;
- Андрей Василевский (Tampa Bay) - в матче 1:0 в овертайме против Montréal;
- **Brandon Bussi** (Carolina) - в последнем матче финала (3:0 у Vegas);
- **Joonas Korpisalo** (Boston) и **Scott Wedgewood** (Colorado) выходили на замену и не пропустили, пока были на льду. Korpisalo заменил Jeremy Swayman в третьем периоде, когда Boston уже проигрывал 0:6. Wedgewood заменил Mackenzie Blackwood после трёх пропущенных голов в первом периоде, после чего Colorado отыгралась и победила Minnesota в овертайме 4:3.

Если бы мы не восстановили вратарей у победных голов в овертайме в самом начале, в список ошибочно попали бы ещё вратари, проигравшие в овертайме : в данных у этих голов нет вратаря.

## 22. Собственный вопрос: comeback is real (?)

Как часто в плей-офф команды отыгрываются после отставания в две шайбы и больше? Какой камбэк был самым большим?

В хоккее отставание в 2 шайбы считается серьёзным, особенно в плей-офф. Проверим, насколько оно на самом деле «смертельно», и найти матчи, где команда смогла вырвать победу из безнадёжной ситуации.

1. Для каждого события посчитаем разницу в счёте с точки зрения хозяев: `Score Diff` = голы хозяев − голы гостей (по полям `Cumulated Home Score` и `Cumulated Away Score`).
2. Для каждого матча найдём наибольшее отставание хозяев (минимум разницы со знаком минус) и наибольшее отставание гостей (максимум разницы). Каждая игра начинается при счёте 0:0, поэтому минимум разницы всегда не больше 0, а максимум — не меньше 0.
3. Посмотрим, с каким наибольшим отставанием сталкивался победитель каждого матча. Если оно 2 и больше - это камбэк.
4. Посчитаем, как часто команда, уступавшая 2 шайбы и больше, всё-таки сравнивала счёт и выигрывала.
5. Нарисуем, как менялся счёт в самом большом камбэке.

### Наибольшее отставание в каждом матче

In [ ]:
events['Score Diff'] = events['Cumulated Home Score'] - events['Cumulated Away Score']

games['Home Max Deficit'] = -events.groupby('Game ID')['Score Diff'].min()
games['Away Max Deficit'] = events.groupby('Game ID')['Score Diff'].max()

home_won = games['Home Score'] > games['Away Score']

games['Winner Max Deficit'] = games['Away Max Deficit']
games.loc[home_won, 'Winner Max Deficit'] = games.loc[home_won, 'Home Max Deficit']

games[['Game Info', 'Home Max Deficit', 'Away Max Deficit', 'Winner Max Deficit']].head()

Посмотрим, сколько раз победитель вообще не уступал по ходу матча (0), уступал одну шайбу (1), две (2) и т. д.

In [ ]:
games['Winner Max Deficit'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.countplot(data=games, x='Winner Max Deficit', color='tab:blue')
plt.xlabel('Наибольшее отставание победителя по ходу матча, шайб')
plt.ylabel('Число матчей')
plt.title('С каким отставанием сталкивались победители матчей')
plt.show()

В 46 матчах из 82 - больше половины! - победитель ни разу не уступал по ходу игры: он забил первым и потом не позволял сопернику выйти вперёд. Ещё в 27 матчах победитель уступал одну шайбу и отыгрался. Победа после отставания в 2 шайбы случилась всего 7 раз, а в 3 шайбы - 2 раза. Больше чем с трёх шайб не отыгрывался никто.

### Матчи-камбэки

Выберем матчи, в которых победитель по ходу игры уступал 2 шайбы и больше:

In [ ]:
comebacks = games[games['Winner Max Deficit'] >= 2]
comebacks = comebacks.sort_values('Winner Max Deficit', ascending=False)

pd.set_option('display.max_colwidth', 100)
comebacks[['Game Info', 'Winner Max Deficit']]

Какие команды совершали камбэки:

In [ ]:
comeback_teams = []
for game_id in comebacks.index:
    g = games.loc[game_id]
    if g['Home Score'] > g['Away Score']:
        comeback_teams.append(g['Home Team'])
    else:
        comeback_teams.append(g['Away Team'])

pd.Series(comeback_teams).value_counts()

Всего камбэков (победа после отставания в 2+ шайбы) 9 из 82 матчей, то есть примерно каждый девятый матч. Самые большие камбэки — с 0:3:
- **Vegas Golden Knights — Colorado Avalanche 5:3** (матч 24 мая);
- **Colorado Avalanche — Minnesota Wild 4:3 в овертайме** (второй раунд матча 13 мая).

Два камбэка случились прямо в финале! Самый мощный - у Buffalo в матче против Montréal, который закончился разгромом 8:3, хотя Buffalo сначала уступали 2 шайбы.

Камбэки распределены по командам довольно равномерно: по 2 у Vegas, Carolina и Buffalo, по одному у Colorado, Tampa Bay и Anaheim. Оба финалиста хотя бы дважды выигрывали после отставания в 2 шайбы.

### Насколько опасно отставание в 2 шайбы?

Посмотрим с другой стороны — со стороны команды, которая отставала. Соберём таблицу, где каждая строка - одна команда в одном матче: её наибольшее отставание и выиграла ли она. Для каждого матча в ней две строки: хозяева и гости.

In [ ]:
home_part = pd.DataFrame({'Team': games['Home Team'],
                          'Max Deficit': games['Home Max Deficit'],
                          'Win': home_won})
away_part = pd.DataFrame({'Team': games['Away Team'],
                          'Max Deficit': games['Away Max Deficit'],
                          'Win': ~home_won})
team_games = pd.concat([home_part, away_part])

# для каждого отставания: сколько раз было, сколько побед и доля побед
deficit_table = team_games.groupby('Max Deficit')['Win'].agg(['count', 'sum', 'mean'])
deficit_table.columns = ['Team-games', 'Wins', 'Win Share']
deficit_table

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.barplot(x=deficit_table.index, y=deficit_table['Win Share'], color='tab:blue')
plt.xlabel('Наибольшее отставание команды по ходу матча, шайб')
plt.ylabel('Доля побед')
plt.title('Как часто побеждает команда в зависимости от отставания')
plt.show()

Победить после отставания в 2 шайбы это лучший случай. Но часто команда отыгрывается, то есть сравнивает счёт, а потом всё равно проигрывает. Чтобы это посчитать, нужно смотреть на порядок событий: сначала отставание 2+, потом счёт снова равный. Сделаем это простым циклом по событиям каждого матча.

In [ ]:
home_equalized = []
away_equalized = []

for game_id in games.index:
    diffs = events[events['Game ID'] == game_id]['Score Diff']
    home_down_two = False   # хозяева уже отставали на 2+
    away_down_two = False   # гости уже отставали на 2+
    home_tied = False       # хозяева сравняли счёт после такого отставания
    away_tied = False
    for d in diffs:
        if d <= -2:
            home_down_two = True
        if d >= 2:
            away_down_two = True
        if d == 0 and home_down_two:
            home_tied = True
        if d == 0 and away_down_two:
            away_tied = True
    home_equalized.append(home_tied)
    away_equalized.append(away_tied)

games['Home Equalized'] = home_equalized
games['Away Equalized'] = away_equalized

trailing_two = team_games[team_games['Max Deficit'] >= 2]
n_trailing = len(trailing_two)
n_equalized = games['Home Equalized'].sum() + games['Away Equalized'].sum()
n_won = trailing_two['Win'].sum()

print('Случаев, когда команда отставала на 2+ шайбы:', n_trailing)
print('Из них сравняли счёт:', n_equalized, '({:.1%})'.format(n_equalized / n_trailing))
print('Из них выиграли матч:', n_won, '({:.1%})'.format(n_won / n_trailing))

**Комментарии к таблице и графикам:**
- Команда, которая ни разу не уступала (отставание 0), выиграла во всех 46 случаях. Это логично: в плей-офф ничьих нет, и если команда ни разу не уступала, то победный гол обязательно её.
- С ростом отставания шансы быстро падают. Если команда уступала максимум одну шайбу, она выигрывала примерно в половине случаев. При отставании в 2 шайбы только в 23 % случаев, в 3 шайбы в 10 %. Ни одна команда, уступавшая 4 шайбы и больше, не победила.
- Всего команды уступали 2 шайбы и больше 67 раз. Сравнять счёт удалось в 19 случаях, а выиграть - в 9. Получается, что из 19 команд, которые отыгрались до ничьей, победили 9 - примерно половина. После того как счёт сравнялся, шансы снова становятся примерно равными, а главная трудность - именно отыграть две шайбы.

### Ответ на вопрос:

- Отставание в 2 шайбы в плей-офф НХЛ очень опасно, но не безнадёжно. Из 67 таких случаев команда сравнивала счёт в 28 %, а побеждала в 13 % (примерно каждая седьмая-восьмая команда).
- Камбэки случились в 9 матчах из 82. Самый большой — с 0:3, таких было два: Vegas — Colorado 5:3 и Colorado — Minnesota 4:3 в овертайме. После отставания в 4 шайбы и больше не отыгрался никто.
- Чаще всего победитель матча вообще ни разу не уступал (56 % матчей), поэтому первый гол и игра «от счёта» очень важны в плей-офф.
- Если команде удаётся сравнять счёт после отставания в 2 шайбы, дальше шансы примерно равные: половина таких команд в итоге побеждает.

**НО!** Матчей всего 82, поэтому доли посчитаны по маленьким выборкам (например, отставание в 3 шайбы было всего 21 раз) и в другом сезоне могут заметно отличаться. Кроме того, мы не учитываем, когда было отставание: уступать 2 шайбы в начале матча и за 5 минут до конца - разные ситуации.


## Общие выводы

1. В плей-офф НХЛ 2025/2026 было сыграно 82 матча, в среднем с 5.9 гола за игру. Больше четверти матчей потребовали овертайма. Обладателем Кубка стала Carolina Hurricanes, обыгравшая в финале Vegas Golden Knights 4–2 в серии.
2. Большинство бросков делается из зоны нападения с 10–60 футов. Дальние броски со своей половины почти всегда записаны как точные - это в основном броски в пустые ворота в конце матча и вбросы шайбы в зону.
3. Доля бросков в створ у команд мало отличается (60–70 %) и почти не связана с победой. Число бросков в створ связано с числом голов умеренно (корреляция 0.52).
4. Индивидуальные лидеры: по голам - Brett Howden (14), по передачам - Jack Eichel (20), по очкам - Mitch Marner (29), все из Vegas. Больше всех сухих матчей у Frederik Andersen (Carolina, 3).
5. Отставание в 2 шайбы оставляет команде примерно 13 % шансов на победу. Самые большие камбэки в этом плей-офф - с 0:3.